# Persistence baseline

Test whether carrying the latest displacement forward provides a useful reference forecast on an explicit storm split.

In [ ]:
from pathlib import Path
import pandas as pd
import tomllib

from storm_tracks.data import split_by_storm, validate_disjoint_storm_splits
from storm_tracks.evaluation import evaluate_persistence, summarize_lead_time_errors

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
with (PROJECT_ROOT / "configs" / "pipeline.toml").open("rb") as file:
    config = tomllib.load(file)
tracks = pd.read_pickle(PROJECT_ROOT / config["paths"]["processed_data"])

## Split storms without overlap

In [ ]:
splits = split_by_storm(
    tracks,
    train_fraction=config["splits"]["train_fraction"],
    validation_fraction=config["splits"]["validation_fraction"],
    seed=config["splits"]["seed"],
)
validate_disjoint_storm_splits(
    train=splits.train, validation=splits.validation, test=splits.test
)
split_name = config["splits"]["evaluation"]
evaluation_tracks = getattr(splits, split_name)

## Evaluate the baseline

In [ ]:
persistence_results = evaluate_persistence(
    evaluation_tracks,
    evaluation_tracks["sid"].unique(),
    split=split_name,
    lead_times_hours=config["forecast"]["lead_times_hours"],
    min_origin_index=config["forecast"]["min_origin_index"],
    max_origins_per_storm=config["forecast"]["max_origins_per_storm"],
)
summarize_lead_time_errors(persistence_results, split=split_name)

## Compare models

The Kalman filter notebook uses the same split and forecast horizons so its errors can be compared directly with this baseline.